# 🔄 Telethon Session to Telegram Desktop Converter
## Google Colab Edition - No Installation Required!

### 📋 Instructions:
1. Run each cell in order (click the play button)
2. Upload your `.session` file when prompted
3. Enter your API credentials
4. Download the generated `tdata` folder

### ⏱️ Note on Keep-Alive:
Colab disconnects after 12 hours of inactivity. For the full 24-hour keep-alive:
- Option A: Reconnect after 12h and resume
- Option B: Use Oracle Cloud or WSL for uninterrupted 24h
- Option C: Just let it run - you'll still get admin rights eventually

---

## Step 1: Install Dependencies
This installs all required libraries. Takes ~30 seconds.

In [ ]:
!pip install -q telethon opentele tgcrypto
print("✅ All dependencies installed successfully!")

## Step 2: Upload Your Session File
Click "Choose Files" and select your `.session` file

In [ ]:
from google.colab import files
import os

print("📤 Please upload your .session file:")
uploaded = files.upload()

# Get the session filename
session_files = [f for f in uploaded.keys() if f.endswith('.session')]

if not session_files:
    print("❌ Error: No .session file uploaded!")
else:
    SESSION_FILE = session_files[0]
    SESSION_NAME = SESSION_FILE.replace('.session', '')
    print(f"✅ Session file uploaded: {SESSION_FILE}")
    print(f"   Session name: {SESSION_NAME}")

## Step 3: Configure API Credentials
Get your API credentials from: https://my.telegram.org/apps

In [ ]:
import getpass

print("🔑 Enter your Telegram API credentials")
print("   Get them from: https://my.telegram.org/apps\n")

API_ID = int(input("Enter your API ID (number): "))
API_HASH = getpass.getpass("Enter your API Hash: ")

print("\n✅ Credentials configured!")

## Step 4: Convert Session to tdata
This will convert your Telethon session to Telegram Desktop format.

In [ ]:
import asyncio
from opentele.td import TDesktop
from opentele.tl import TelegramClient
from opentele.api import API, UseCurrentSession

async def convert_session():
    print("🔄 Starting conversion...\n")
    
    # Create client
    client = TelegramClient(SESSION_NAME, API_ID, API_HASH)
    await client.connect()
    
    if not await client.is_user_authorized():
        print("❌ Session is not authorized!")
        return None
    
    # Get user info
    me = await client.get_me()
    print(f"✅ Logged in as: {me.first_name} (@{me.username})")
    print(f"   Phone: {me.phone}\n")
    
    # Convert to tdata
    print("🔄 Converting to Telegram Desktop format...")
    tdesk = await client.ToTDesktop(
        flag=UseCurrentSession,
        api=API.TelegramDesktop
    )
    
    # Save tdata
    output_dir = "./output_tdata"
    os.makedirs(output_dir, exist_ok=True)
    tdesk.SaveTData(output_dir)
    
    print(f"✅ Conversion complete!")
    print(f"   Output directory: {output_dir}\n")
    
    return client

# Run conversion
client = await convert_session()

## Step 5: Monitor Active Sessions
View all active sessions on your account (including that Moldova one!)

In [ ]:
from telethon.tl.functions.account import GetAuthorizationsRequest

async def list_sessions():
    if client is None:
        print("❌ No active client. Run Step 4 first.")
        return
    
    auths = await client(GetAuthorizationsRequest())
    
    print("=" * 70)
    print(f"ACTIVE SESSIONS: {len(auths.authorizations)}")
    print("=" * 70 + "\n")
    
    for i, auth in enumerate(auths.authorizations, 1):
        print(f"[Session {i}]")
        print(f"  Device: {auth.device_model}")
        print(f"  Platform: {auth.platform} {auth.system_version}")
        print(f"  App: {auth.app_name} {auth.app_version}")
        print(f"  Location: {auth.country}, {auth.region}")
        print(f"  IP: {auth.ip}")
        print(f"  Current: {'YES ✅' if auth.current else 'NO'}")
        print(f"  Official: {'YES' if auth.official_app else 'NO'}")
        
        # Highlight suspicious
        if "moldova" in auth.country.lower() and not auth.current:
            print(f"  ⚠️  SUSPICIOUS SESSION FROM MOLDOVA!")
        
        print()
    
    print("=" * 70)

await list_sessions()

## Step 6: Keep-Alive Loop (Optional)
Keep your session active for 24 hours to gain admin privileges.

**Note:** Colab may disconnect after 12 hours. To resume:
1. Reconnect to the runtime
2. Re-run this cell
3. Continue from where you left off

In [ ]:
from telethon.tl.functions.updates import GetStateRequest
from datetime import datetime, timedelta
import time

# Configuration
KEEP_ALIVE_HOURS = 24
PING_INTERVAL_SECONDS = 300  # 5 minutes

async def keep_alive():
    if client is None:
        print("❌ No active client. Run Step 4 first.")
        return
    
    start_time = datetime.now()
    end_time = start_time + timedelta(hours=KEEP_ALIVE_HOURS)
    ping_count = 0
    
    print("=" * 70)
    print("⏰ KEEP-ALIVE MODE ACTIVATED")
    print(f"Started: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Target: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Duration: {KEEP_ALIVE_HOURS} hours")
    print("=" * 70 + "\n")
    
    print("💡 TIP: Keep this tab open. If Colab disconnects:")
    print("   1. Reconnect to runtime")
    print("   2. Re-run this cell to continue\n")
    
    try:
        while datetime.now() < end_time:
            try:
                # Ping Telegram
                await client(GetStateRequest())
                ping_count += 1
                
                elapsed = datetime.now() - start_time
                remaining = end_time - datetime.now()
                
                print(f"[Ping #{ping_count}] {datetime.now().strftime('%H:%M:%S')} | "
                      f"Elapsed: {str(elapsed).split('.')[0]} | "
                      f"Remaining: {str(remaining).split('.')[0]}")
                
                # Check sessions every hour
                if ping_count % (3600 // PING_INTERVAL_SECONDS) == 0:
                    print("\n📊 Checking active sessions...\n")
                    await list_sessions()
                
                # Wait before next ping
                await asyncio.sleep(PING_INTERVAL_SECONDS)
                
            except Exception as e:
                print(f"⚠️  Ping failed: {e}")
                if not client.is_connected():
                    print("🔄 Reconnecting...")
                    await client.connect()
                await asyncio.sleep(10)
        
        print("\n" + "=" * 70)
        print("✅ KEEP-ALIVE COMPLETED!")
        print(f"Total pings: {ping_count}")
        print("Your session should now have administrative privileges.")
        print("You can now terminate suspicious sessions!")
        print("=" * 70)
        
    except KeyboardInterrupt:
        print("\n⚠️  Keep-alive interrupted by user")
        print(f"Total pings: {ping_count}")

# Ask user if they want to run keep-alive
run_keepalive = input("\nRun 24-hour keep-alive? (yes/no): ").lower().strip()

if run_keepalive in ['yes', 'y']:
    await keep_alive()
else:
    print("⏭️  Skipping keep-alive. Proceeding to download...")

## Step 7: Download Your tdata Folder
Download the generated tdata to use with Telegram Desktop

In [ ]:
import shutil

# Create a zip file
print("📦 Creating zip archive of tdata...")
shutil.make_archive('tdata_output', 'zip', './output_tdata')

print("✅ Archive created!")
print("\n📥 Downloading tdata_output.zip...")

# Download the file
files.download('tdata_output.zip')

print("\n" + "=" * 70)
print("🎉 SUCCESS! Your tdata is ready!")
print("=" * 70)
print("\nNext steps:")
print("1. Extract tdata_output.zip")
print("2. Close Telegram Desktop if running")
print("3. Replace your tdata folder with the extracted one")
print("4. Launch Telegram Desktop - you'll be logged in!")
print("\n💡 After 24 hours, you can terminate that Moldova session from:")
print("   Settings → Privacy & Security → Active Sessions")
print("=" * 70)

## Step 8: Cleanup (Optional)
Disconnect and clean up temporary files

In [ ]:
# Disconnect client
if client:
    await client.disconnect()
    print("✅ Client disconnected")

# Remove session file for security
import os
if os.path.exists(SESSION_FILE):
    os.remove(SESSION_FILE)
    print("✅ Session file removed")

print("\n🧹 Cleanup complete!")

---

## 📚 Additional Information

### About the Moldova Session
The suspicious session will be highlighted in Step 5. After your new session matures (24 hours), you can:
1. Open Telegram Desktop with your new tdata
2. Go to Settings → Privacy & Security → Active Sessions
3. Find the Moldova session
4. Click "Terminate"

### Colab Limitations
- Maximum 12 hours continuous runtime
- For full 24h keep-alive, you'll need to:
  - Reconnect after 12h
  - Re-run Step 6 to continue
  
### Alternative Platforms
For uninterrupted 24-hour keep-alive, consider:
- Oracle Cloud (free forever)
- WSL on Windows
- DigitalOcean ($0.50 for 24h)

See `PLATFORM_ALTERNATIVES.md` for detailed guides.

---

### ⚠️ Security Notes
- Never share your .session files
- Keep API credentials private
- Enable 2FA on your Telegram account
- Regularly check active sessions

---

### 💬 Need Help?
If you encounter issues:
1. Check the error messages in the cell outputs
2. Verify your API credentials
3. Make sure your session file is valid
4. Try reconnecting to the Colab runtime

---

**Made with ❤️ for secure Telegram session management**